In [1]:
%pip install -q diffusers transformers accelerate safetensors sentencepiece "pandas==2.2.3" "pillow>=11,<13" "protobuf>=5.29.1,<7" matplotlib

In [2]:
import sys
import json
import platform
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version

import torch
from diffusers import AutoPipelineForInpainting

assert torch.cuda.is_available(), (
    "Enable a GPU under Runtime → Change runtime type."
)

ROOT = Path("/content/comp8410_image_editing")

for folder in ["inputs", "masks", "outputs", "results"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

environment = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gb": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ),
    "packages": {
        name: version(name)
        for name in [
            "torch", "diffusers", "transformers",
            "accelerate", "safetensors", "pillow"
        ]
    },
}

(ROOT / "results" / "environment.json").write_text(
    json.dumps(environment, indent=2)
)

requirements = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)
(ROOT / "results" / "environment_freeze.txt").write_text(requirements)

print(json.dumps(environment, indent=2))
print("\nSetup passed. Experiment folder:", ROOT)

{
  "recorded_at_utc": "2026-10-30T07:49:07.505895+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "gpu": "Tesla T4",
  "gpu_memory_gb": 14.56,
  "packages": {
    "torch": "2.11.0+cu130",
    "diffusers": "0.42.1",
    "transformers": "5.22.0",
    "accelerate": "1.15.0",
    "safetensors": "0.8.0",
    "pillow": "12.3.0"
  }
}

Setup passed. Experiment folder: /content/comp8410_image_editing


In [4]:
import json
import time
from pathlib import Path

import torch
from diffusers import AutoPipelineForInpainting

assert torch.cuda.is_available(), (
    "Enable a GPU under Runtime → Change runtime type."
)

ROOT = Path("/content/comp8410_image_editing")
(ROOT / "results").mkdir(parents=True, exist_ok=True)

MODEL_A_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"

# Reuse the pipeline if the previous cell loaded it successfully.
reused_pipeline = "pipe_a" in globals() and pipe_a is not None

if not reused_pipeline:
    print("Loading SDXL Inpainting...")
    start = time.perf_counter()

    pipe_a = AutoPipelineForInpainting.from_pretrained(
        MODEL_A_ID,
        dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )

    load_seconds = round(time.perf_counter() - start, 2)
else:
    print("Reusing the model already loaded in this session.")
    load_seconds = None

# Enable CPU offloading unless it was already enabled.
if getattr(pipe_a, "_offload_gpu_id", None) is None:
    pipe_a.enable_model_cpu_offload()

# Apply memory settings directly to the VAE.
pipe_a.vae.enable_slicing()
pipe_a.vae.enable_tiling()

model_info = {
    "model_id": MODEL_A_ID,
    "pipeline_class": type(pipe_a).__name__,
    "precision": "float16",
    "cpu_offload": True,
    "vae_slicing": True,
    "vae_tiling": True,
    "reused_pipeline": reused_pipeline,
    "load_seconds": load_seconds,
    "gpu": torch.cuda.get_device_name(0),
    "status": "loaded; inference not yet tested",
}

setup_path = ROOT / "results" / "model_a_setup.json"
setup_path.write_text(json.dumps(model_info, indent=2))

print(json.dumps(model_info, indent=2))
print("\nModel A is ready for the first image test.")

Reusing the model already loaded in this session.
{
  "model_id": "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
  "pipeline_class": "StableDiffusionXLInpaintPipeline",
  "precision": "float16",
  "cpu_offload": true,
  "vae_slicing": true,
  "vae_tiling": true,
  "reused_pipeline": true,
  "load_seconds": null,
  "gpu": "Tesla T4",
  "status": "loaded; inference not yet tested"
}

Model A is ready for the first image test.


image needed

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from IPython.display import display
import io

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one product photo.")

filename, image_bytes = next(iter(uploaded.items()))

# Correct phone-camera orientation and preserve the original.
original_image = ImageOps.exif_transpose(
    Image.open(io.BytesIO(image_bytes))
).convert("RGB")

(ROOT / "inputs").mkdir(parents=True, exist_ok=True)
original_image.save(ROOT / "inputs" / "pilot_original.png")

# Preserve proportions and pad to a square for the pilot.
pilot_image = ImageOps.pad(
    original_image,
    (768, 768),
    method=Image.Resampling.LANCZOS,
    color="white",
)
pilot_image.save(ROOT / "inputs" / "pilot_input.png")

print("Uploaded:", filename)
print("Original size:", original_image.size)
print("Pilot size:", pilot_image.size)
display(pilot_image)